[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C43_Data_Engineering_Course/03_tokenization_throughput/03_tokenization_throughput.ipynb)

# 03 · Tokenization 吞吐（用 numpy/标准库从零实现 + 算账）

目标：把 **吞吐账 → 序列 packing(浪费率) → 文档掩码(防跨文档污染) → Amdahl 并行加速 → token 内存账** 从零实现，每个机制都跑出**可打印的验证**与 `assert`，每笔账都推演到 **万亿 token** 规模。

路线：吞吐账 → 朴素 padding 的浪费 → 贪心 packing → 块对角文档掩码 → Amdahl 加速比 → uint16/uint32 内存账 → ✏️ 练习 → 📖 答案 → 🧪 Megatron/SentencePiece 胶囊。

> 心智模型：**分词易并行（逐文档独立），但 Amdahl 封顶；padding 是隐形浪费，packing 消灭它（但要配掩码）；token 存几字节由词表定**。我们写机制*结构*与*正确性*，规模由账目推演。

## 1 · 吞吐账：万亿 token 分词要多久

分词时间 ≈ `N / (R · P · η)`：N 总 token，R 单核 tok/s，P 核数，η 并行效率。
先把这笔账算出来——它把「要不要并行、并行到什么程度」从拍脑袋变成可规划的工程决策。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def tokenize_seconds(total_tokens, per_core_tok_s, cores, eff=0.8):
    '''分完所需秒数。eff<1 反映 IO/调度/汇总损耗。'''
    return total_tokens / (per_core_tok_s * cores * eff)

N = 1e12            # 1 万亿 token
R = 1e5             # 单核 ~10 万 tok/s（保守，典型量级）
print(f"{'核数 P':>8s} {'有效吞吐(tok/s)':>18s} {'分完 1e12':>14s}")
for P in [1, 64, 1024, 10000]:
    secs = tokenize_seconds(N, R, P)
    days = secs / 86400
    human = f'{secs/3600:.1f} 小时' if days < 1 else f'{days:.1f} 天'
    print(f'{P:>8d} {R*P*0.8:>18.2e} {human:>14s}')

secs_1 = tokenize_seconds(N, R, 1)
secs_1024 = tokenize_seconds(N, R, 1024)
assert secs_1 / 86400 > 100, '单核应是百天级 -> 必须并行'
assert secs_1 / secs_1024 > 900, '1024 核应带来近千倍加速（线性段）'
print('\n✅ 吞吐账：单核 116 天，1024 核 3.4 小时。分词必须并行 —— 但下一节看到并行有上限。')

## 2 · 朴素 padding 的浪费有多大

训练吃**定长** L 的序列。朴素做法：每篇文档单独成条、短的填 padding 到 L。
浪费率 = padding token 数 / 总 token 数。文档越短于 L，浪费越触目惊心。

In [ ]:
def naive_padding_waste(doc_lengths, L):
    '''每篇文档单独占一条长度 L 的序列(超长则截断到 L)。返回 (浪费率, 总格子数, 真实token数)。'''
    doc_lengths = [min(d, L) for d in doc_lengths]      # 超长截断
    n_seqs = len(doc_lengths)
    total_slots = n_seqs * L
    real = sum(doc_lengths)
    waste = 1 - real / total_slots
    return waste, total_slots, real

L = 2048
# 真实语料文档长度高度偏斜：大量短文档 + 少量长文档（对数正态近似）
doc_lengths = np.clip(rng.lognormal(mean=5.0, sigma=1.2, size=2000).astype(int), 1, 8000)
print(f'文档数={len(doc_lengths)}, 平均长度={doc_lengths.mean():.0f}, 中位={np.median(doc_lengths):.0f}, 序列长 L={L}')
waste, slots, real = naive_padding_waste(doc_lengths, L)
print(f'朴素 padding：{slots:,} 个格子，只有 {real:,} 个真实 token')
print(f'-> padding 浪费率 = {waste:.1%}（这些算力全烧在填充符上！）')
assert waste > 0.5, '短文档为主时朴素 padding 浪费应很高'
print('✅ 朴素 padding 浪费惊人：平均文档远短于 L 时，大半算力在算填充符')

## 3 · 序列 packing：贪心装箱消灭浪费

把多篇短文档**拼进同一条** L 长序列，几乎不留空。这是个**装箱(bin packing)** 问题(NP 难)，
用贪心近似即可：**首次适配(first-fit)** —— 来一篇放进第一个装得下的箱子，装不下就开新箱。
目标：浪费率从 90% 砍到接近 0。

In [ ]:
def pack_first_fit(doc_lengths, L):
    '''首次适配装箱：返回 bins(每个是文档长度列表) 与浪费率。超长文档截断到 L 独占一箱。'''
    bins = []              # 每个 bin 是 [文档长度, ...]，sum<=L
    bin_used = []          # 每个 bin 已用长度
    for d in doc_lengths:
        d = min(d, L)
        placed = False
        for k in range(len(bins)):
            if bin_used[k] + d <= L:       # 第一个装得下的箱
                bins[k].append(d); bin_used[k] += d; placed = True; break
        if not placed:
            bins.append([d]); bin_used.append(d)
    total_slots = len(bins) * L
    real = sum(bin_used)
    waste = 1 - real / total_slots
    return bins, waste

bins, waste_packed = pack_first_fit(list(doc_lengths), L)
print(f'朴素 padding：{len(doc_lengths):,} 条序列，浪费 {waste:.1%}')
print(f'packing 后  ：{len(bins):,} 条序列，浪费 {waste_packed:.1%}')
print(f'-> 序列数减少 {len(doc_lengths)/len(bins):.1f}x，浪费率从 {waste:.0%} 降到 {waste_packed:.0%}')
assert waste_packed < 0.1, 'packing 后浪费率应很低'
assert len(bins) < len(doc_lengths), 'packing 应显著减少序列条数'
print('✅ packing 把浪费打到个位数 % —— 有效算力近乎翻倍')

## 4 · 文档掩码：packing 的正确性前提

一条序列拼了多篇文档，**必须**用块对角注意力掩码阻止注意力**跨文档边界**，
否则第二篇会「看到」第一篇 —— **跨文档污染**(Krell 2021)。

构造 `doc_ids`（每个位置属于哪篇文档），掩码 `M[i,j]=1` 当且仅当 i,j 同文档**且** j≤i（因果）。

In [ ]:
def build_packed_sequence(bin_docs, L, sep=0):
    '''把一个 bin(文档长度列表) 摊平成长度 L 的序列，返回 (tokens占位, doc_ids)。
       doc_ids[k]=该位置属于第几篇文档；padding 位 doc_ids=-1。'''
    doc_ids = np.full(L, -1, dtype=np.int64)
    pos = 0
    for did, dlen in enumerate(bin_docs):
        doc_ids[pos:pos+dlen] = did
        pos += dlen
    return doc_ids

def block_diag_causal_mask(doc_ids):
    '''返回 (L,L) 掩码：M[i,j]=True 可注意 当 同文档 且 j<=i 且 都非 padding。'''
    L = len(doc_ids)
    same_doc = (doc_ids[:, None] == doc_ids[None, :])     # 同文档
    causal = np.tril(np.ones((L, L), dtype=bool))         # j<=i
    valid = (doc_ids[:, None] >= 0) & (doc_ids[None, :] >= 0)
    return same_doc & causal & valid

# 用一个小 bin 演示：三篇文档长度 3,2,4 拼进 L=12
doc_ids = build_packed_sequence([3, 2, 4], L=12)
M = block_diag_causal_mask(doc_ids)
print('doc_ids:', doc_ids)
# 关键验证：文档 0 的位置(0,1,2) 绝不能注意到文档 1 的位置(3,4)
assert not M[2, 3], '文档1的token不该被文档0看到（跨文档泄漏！）'
assert not M[5, 0], '后一篇文档不该注意到前一篇（跨文档泄漏！）'
assert M[2, 0] and M[2, 1] and M[2, 2], '同文档内因果注意应允许'
assert not M[0, 2], '因果：位置0不该看到未来的位置2'
# padding 位(9,10,11) 完全屏蔽
assert not M[9].any() and not M[:, 9].any(), 'padding 位应全屏蔽'
print('✅ 块对角因果掩码正确：各文档严格隔离，无跨文档泄漏 —— packing 的正确性前提')

## 5 · Amdahl 定律：并行加速的天花板

分词编码可并行，但读 IO、写盘汇总、BPE 合并循环有**串行占比 s**。
加速比上限 `Speedup(P) = 1/(s + (1-s)/P)`，当 P→∞ 时被 **1/s** 死死封住。

先验证这条曲线，再看「加核到某点后收益趋零」。

In [ ]:
def amdahl_speedup(s, P):
    '''串行占比 s，P 核，返回加速比。'''
    return 1.0 / (s + (1.0 - s) / P)

print(f"{'核数 P':>8s} {'s=1%':>8s} {'s=5%':>8s} {'s=20%':>8s}")
for P in [1, 10, 100, 1000, 100000]:
    print(f'{P:>8d} {amdahl_speedup(0.01,P):>8.1f} {amdahl_speedup(0.05,P):>8.1f} {amdahl_speedup(0.20,P):>8.1f}')

# 上限 = 1/s，与核数无关
assert abs(amdahl_speedup(0.05, 10**9) - 20.0) < 0.1, 's=5% 的加速上限是 20x'
assert amdahl_speedup(0.20, 10**9) < 5.5, 's=20% 上限是 5x，再多核也没用'
# 收益递减：1000->100000 核，s=5% 时加速几乎不再增长
gain = amdahl_speedup(0.05, 100000) / amdahl_speedup(0.05, 1000)
assert gain < 1.05, '串行占比下，加核 100 倍收益<5% -> 该治串行而非加核'
print('\n✅ Amdahl：s=5% 时无论多少核最多 20x。加核前必须先量化、压低串行占比 s。')

## 6 · 内存账：token 用 uint16 还是 uint32

token id ∈ [0, vocab)。`vocab < 65536` 用 **uint16(2B)** 够；更大词表用 **uint32(4B)**。
在万亿 token 上是 **2TB vs 4TB** 的差别 —— 直接关系存储成本与训练 IO 带宽。

In [ ]:
def token_dtype(vocab_size):
    '''按词表大小选最省的无符号整型。'''
    if vocab_size <= 2**8:  return np.uint8, 1
    if vocab_size <= 2**16: return np.uint16, 2
    if vocab_size <= 2**32: return np.uint32, 4
    return np.uint64, 8

def corpus_storage_bytes(n_tokens, vocab_size):
    _, nbytes = token_dtype(vocab_size)
    return n_tokens * nbytes

N = int(1e12)
print(f"{'词表大小':>12s} {'dtype':>10s} {'1e12 token 落盘':>18s}")
for V in [50257, 65536, 128000, 256000]:    # GPT-2 / 临界 / 多语种 / 超大
    dt, nb = token_dtype(V)
    tb = corpus_storage_bytes(N, V) / 1e12
    print(f'{V:>12,d} {dt.__name__:>10s} {tb:>15.1f} TB')

# GPT-2 词表 50257 < 65536 -> uint16
dt_small, nb_small = token_dtype(50257)
assert dt_small == np.uint16 and nb_small == 2
# 12.8 万多语种词表 > 65536 -> 被迫 uint32，存储翻倍
dt_big, nb_big = token_dtype(128000)
assert dt_big == np.uint32 and nb_big == 4
assert corpus_storage_bytes(N, 128000) == 2 * corpus_storage_bytes(N, 50257)
# 真把一小段 token 用对的 dtype 存下来，验证字节数
toks = rng.integers(0, 50257, size=1000).astype(dt_small)
assert toks.nbytes == 2000, 'uint16 存 1000 token = 2000 字节'
print('\n✅ 内存账：词表跨过 65536 就被迫 uint32、存储翻倍。先问 dtype，再乘总量。')

---
## ✏️ 练习 1：best-fit 装箱（比 first-fit 更省）

实现 `pack_best_fit(doc_lengths, L)`：**最佳适配** —— 来一篇文档，放进
**剩余空间最小但仍装得下**的箱子（比 first-fit 更紧）；都装不下才开新箱。
返回 `(bins, 浪费率)`，bins 是每箱的文档长度列表。

In [ ]:
def pack_best_fit(doc_lengths, L):
    bins, bin_used = [], []
    # TODO: 对每篇 d=min(d,L)，在所有 bin_used[k]+d<=L 的箱里
    #       选「剩余空间 L-bin_used[k] 最小」的那个放入；都不行则开新箱。
    #       最后算 waste = 1 - sum(bin_used)/(len(bins)*L)，返回 (bins, waste)。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
dls = [1500, 600, 500, 400, 300, 200]
bins_bf, waste_bf = pack_best_fit(dls, L=2048)
# 所有文档都被放置（数量守恒）
assert sum(len(b) for b in bins_bf) == len(dls), '每篇文档都要被装箱'
# 每个箱不超过 L
assert all(sum(b) <= 2048 for b in bins_bf), '箱内总长不能超过 L'
# best-fit 不劣于 first-fit（箱数 <=）
bins_ff, _ = pack_first_fit(dls, L=2048)
assert len(bins_bf) <= len(bins_ff), 'best-fit 的箱数应 <= first-fit'
assert 0 <= waste_bf < 1
print(f'best-fit: {len(bins_bf)} 箱, 浪费 {waste_bf:.1%}；first-fit: {len(bins_ff)} 箱')
print('✅ 练习 1 通过：best-fit 装箱正确且不劣于 first-fit')

## ✏️ 练习 2：有效 token 吞吐（packing 的下游收益）

packing 省的算力体现在**有效 token 吞吐**上。给定每秒能处理 `slots_per_s` 个序列格子的硬件，
实现 `effective_token_throughput(slots_per_s, waste)`：返回**每秒真实(非padding) token 数** = `slots_per_s * (1 - waste)`。
再用它对比朴素 padding 与 packing 的有效吞吐差距。

In [ ]:
def effective_token_throughput(slots_per_s, waste):
    # TODO: 返回 slots_per_s * (1 - waste)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
SLOTS = 1e6                       # 硬件每秒处理 1e6 个格子
eff_naive  = effective_token_throughput(SLOTS, waste)         # 朴素(高浪费)
eff_packed = effective_token_throughput(SLOTS, waste_packed)  # packing(低浪费)
assert abs(eff_packed - SLOTS*(1-waste_packed)) < 1e-6
# packing 的有效吞吐应远高于朴素（因为浪费低得多）
assert eff_packed > eff_naive * 1.5, 'packing 应大幅提高有效 token 吞吐'
print(f'朴素有效吞吐 {eff_naive:.2e} tok/s；packing {eff_packed:.2e} tok/s')
print(f'-> packing 让有效算力提升 {eff_packed/eff_naive:.1f}x（同样硬件，多训这么多真实 token）')
print('✅ 练习 2 通过：packing 的收益最终兑现为下游(C08)的有效 token 吞吐')

## ✏️ 练习 3：Amdahl 反推 —— 要达到目标加速比，串行占比最多多少

实现 `max_serial_fraction(target_speedup, P)`：给定核数 `P` 与想要的加速比 `target_speedup`，
反解出**允许的最大串行占比 s**（超过它就达不到目标）。
由 `target = 1/(s + (1-s)/P)` 解出 `s`。

In [ ]:
def max_serial_fraction(target_speedup, P):
    # TODO: 由 target = 1/(s + (1-s)/P) 解 s：
    #   1/target = s + (1-s)/P = s(1 - 1/P) + 1/P
    #   s = (1/target - 1/P) / (1 - 1/P)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
s_max = max_serial_fraction(target_speedup=50, P=1024)
# 反代回 amdahl 应得到约 50x
assert abs(amdahl_speedup(s_max, 1024) - 50) < 0.5, '反推的 s 代回应得到目标加速比'
assert 0 < s_max < 0.05, '要 1024 核达 50x，串行占比必须很小'
# 目标越高，允许的串行占比越小
s_hi = max_serial_fraction(100, 1024)
s_lo = max_serial_fraction(20, 1024)
assert s_hi < s_lo, '更高的目标加速比要求更低的串行占比'
print(f'1024 核要达 50x，串行占比最多 {s_max:.4f} ({s_max:.2%})')
print('✅ 练习 3 通过：Amdahl 反推给出「串行预算」—— 规划并行管线的硬约束')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def pack_best_fit(doc_lengths, L):
    bins, bin_used = [], []
    for d in doc_lengths:
        d = min(d, L)
        best_k, best_rem = -1, L + 1
        for k in range(len(bins)):
            rem = L - bin_used[k]
            if d <= rem and rem < best_rem:      # 装得下且剩余空间最小
                best_k, best_rem = k, rem
        if best_k >= 0:
            bins[best_k].append(d); bin_used[best_k] += d
        else:
            bins.append([d]); bin_used.append(d)
    waste = 1 - sum(bin_used) / (len(bins) * L)
    return bins, waste

In [ ]:
# 练习 2 参考答案
def effective_token_throughput(slots_per_s, waste):
    return slots_per_s * (1 - waste)

In [ ]:
# 练习 3 参考答案
def max_serial_fraction(target_speedup, P):
    return (1.0 / target_speedup - 1.0 / P) / (1.0 - 1.0 / P)

---
## 🧪 真实数据胶囊：Megatron / SentencePiece 数据预处理算账

用真实框架的配置（GPT-NeoX/Megatron 的 `.bin/.idx`、真实词表大小、真实 token 数），算分词管线的三笔账：
① 不同词表下 tokenized 语料落盘体量；② packing 省下的有效算力；③ 给定集群分完要多久。

（带 try/except：本环境不联网，直接用内置的真实量级数字。）

In [ ]:
# 真实配置（公开量级，约数）
CONFIGS = {
    'GPT-2 (en)':        dict(vocab=50257,  tokens=3e11),
    'Llama (32k)':       dict(vocab=32000,  tokens=1.5e13),
    'multilingual(128k)':dict(vocab=128000, tokens=1.5e13),
}

def dtype_bytes(vocab):
    return 2 if vocab <= 65536 else 4    # uint16 vs uint32

print(f"{'配置':>20s} {'dtype字节':>8s} {'落盘体量':>12s}")
for name, cfg in CONFIGS.items():
    nb = dtype_bytes(cfg['vocab'])
    tb = cfg['tokens'] * nb / 1e12
    print(f'{name:>20s} {nb:>8d} {tb:>9.1f} TB')

# 关键：128k 词表被迫 uint32，相同 token 数下落盘是 32k 词表的 2 倍
tb_32k  = CONFIGS['Llama (32k)']['tokens'] * dtype_bytes(32000) / 1e12
tb_128k = CONFIGS['multilingual(128k)']['tokens'] * dtype_bytes(128000) / 1e12
assert abs(tb_128k - 2 * tb_32k) < 1.0, '128k 词表(uint32)落盘应是 32k(uint16)的 2 倍'

# 分完 1.5e13 token：1024 核、单核 1e5 tok/s、效率 0.8
secs = 1.5e13 / (1e5 * 1024 * 0.8)
print(f'\n分完 1.5e13 token @1024核: {secs/86400:.1f} 天')
assert secs/86400 > 1, '1.5e13 token 即便千核也要天级 -> 吞吐是真约束'
print('账目结论：词表大小→dtype→存储翻倍；万亿 token 分词必须并行且要算清存储与工期。')

**🧪 胶囊练习**：实现 `packing_savings(n_docs, avg_len, L, vocab, slots_per_s)`：估算某语料**朴素 padding 的序列数**（n_docs，每篇一条）与**理想 packing 的序列数**（≈ ceil(n_docs*avg_len/L)，假设装满），返回 `(naive_seqs, packed_seqs, 序列数缩减倍数)`。

In [ ]:
import math
def packing_savings(n_docs, avg_len, L, vocab=32000, slots_per_s=1e6):
    # TODO: naive_seqs = n_docs；packed_seqs = ceil(n_docs*min(avg_len,L)/L)；
    #       返回 (naive_seqs, packed_seqs, naive_seqs/packed_seqs)
    raise NotImplementedError

In [ ]:
# 自测
naive, packed, ratio = packing_savings(n_docs=10_000_000, avg_len=256, L=2048)
assert naive == 10_000_000
assert packed == math.ceil(10_000_000 * 256 / 2048)   # 装满假设
assert abs(ratio - naive/packed) < 1e-6
# 平均长度 256、L=2048 -> 理论可塞 8 篇/条 -> 序列数缩减约 8x
assert 7 < ratio < 9, '256 长文档塞进 2048 应缩减约 8x 序列'
print(f'朴素 {naive:,} 条 -> packing {packed:,} 条，缩减 {ratio:.1f}x')
print('✅ 胶囊练习通过：packing 的序列数缩减 ≈ L/平均文档长')

In [ ]:
# 📖 胶囊参考答案
import math
def packing_savings(n_docs, avg_len, L, vocab=32000, slots_per_s=1e6):
    naive_seqs = n_docs
    packed_seqs = math.ceil(n_docs * min(avg_len, L) / L)
    return naive_seqs, packed_seqs, naive_seqs / packed_seqs

---
## 🔧 旁注：真实分词管线长什么样

本课的小模拟，在 Megatron-LM / GPT-NeoX 等真实管线里对应：

- **并行编码**：多进程/多机各分一批文档 shard，用 Rust 实现的 tokenizer（HF tokenizers / tiktoken / SentencePiece）跑 `encode()`，吞吐比纯 Python 快几十倍。
- **packing + 掩码**：把 token 流拼成定长样本，训练框架用块对角/varlen 注意力掩码隔离文档（FlashAttention 的 varlen 接口原生支持，无需显式 padding）。
- **`.bin/.idx` 存储**：所有 token 拼成一个大数组（按词表选 uint16/uint32）存 `.bin`，文档偏移存 `.idx`；训练用 mmap 随机读，无需全量入内存。

你在 numpy 里验证过的 packing 装箱、文档掩码、dtype 选择、Amdahl 估算，可一对一对应到真实管线的工程决策。

### 小结
- 分词逐文档独立、**易并行**，但读 IO/写盘汇总有**串行占比**，**Amdahl 定律**封住加速上限 1/s。
- **吞吐账**：万亿 token 单核要百天，必须并行；加核加到吞吐曲线变平的拐点为止。
- **padding 是隐形浪费**（短文档为主时可达 90%）；**序列 packing**(贪心装箱)消灭它，浪费率→个位数 %。
- packing **必须配文档掩码**（块对角因果），否则**跨文档污染**。
- **内存账**：token 存几字节由词表定（<65536 → uint16，否则 uint32），万亿 token 上是 2TB vs 4TB。

下一站：**模块 04 · 大规模质量过滤** —— 分词前先把垃圾筛掉，怎么「先便宜后贵」地多阶段过滤？